## Dataset Link:- [Kaggle](https://www.kaggle.com/competitions/msk-redefining-cancer-treatment/data)

### Load data

In [34]:
from pathlib import Path
import pandas as pd

# Use .. to go up from 'notebook' to the root, then into 'data/raw'
data_dir = Path('../data/raw') 

# Now your existing code should work
train_data_text = pd.read_csv(data_dir / 'training_text', sep=r'\|\|', engine='python', names=["ID", "TEXT"], skiprows=1)
train_data_variant = pd.read_csv(data_dir / 'training_variants.zip')
test_data_text = pd.read_csv(data_dir / 'test_text', sep=r'\|\|', engine='python', names=["ID", "TEXT"], skiprows=1)
test_data_variant = pd.read_csv(data_dir / 'test_variants.zip')

In [22]:
print(f"[INFO] Training data (variants): {train_data_variant.head()}")
print(f"[INFO] Training data (text): {train_data_text.head()}")
print(f"[INFO] Test data (variants): {test_data_variant.head()}")
print(f"[INFO] Test data (text): {test_data_text.head()}")

[INFO] Training data (variants):    ID    Gene             Variation  Class
0   0  FAM58A  Truncating Mutations      1
1   1     CBL                 W802*      2
2   2     CBL                 Q249E      2
3   3     CBL                 N454D      3
4   4     CBL                 L399V      4
[INFO] Training data (text):    ID                                               TEXT
0   0  Cyclin-dependent kinases (CDKs) regulate a var...
1   1   Abstract Background  Non-small cell lung canc...
2   2   Abstract Background  Non-small cell lung canc...
3   3  Recent evidence has demonstrated that acquired...
4   4  Oncogenic mutations in the monomeric Casitas B...
[INFO] Test data (variants):    ID     Gene Variation
0   0    ACSL4     R570S
1   1    NAGLU     P521L
2   2      PAH     L333F
3   3     ING1     A148D
4   4  TMEM216      G77A
[INFO] Test data (text):    ID                                               TEXT
0   0  2. This mutation resulted in a myeloproliferat...
1   1   Abstract The

## Preprocessing Techniques

### One Hote Encodeing

#### Gene

In [31]:
gene_ohe = pd.get_dummies(train_data_variant["Gene"], prefix="Gene")
print(gene_ohe.head())

   Gene_ABL1  Gene_ACVR1  Gene_AGO2  Gene_AKT1  Gene_AKT2  Gene_AKT3  \
0      False       False      False      False      False      False   
1      False       False      False      False      False      False   
2      False       False      False      False      False      False   
3      False       False      False      False      False      False   
4      False       False      False      False      False      False   

   Gene_ALK  Gene_APC  Gene_AR  Gene_ARAF  ...  Gene_TSC1  Gene_TSC2  \
0     False     False    False      False  ...      False      False   
1     False     False    False      False  ...      False      False   
2     False     False    False      False  ...      False      False   
3     False     False    False      False  ...      False      False   
4     False     False    False      False  ...      False      False   

   Gene_U2AF1  Gene_VEGFA  Gene_VHL  Gene_WHSC1  Gene_WHSC1L1  Gene_XPO1  \
0       False       False     False       False         Fa

#### Variation

In [30]:
var_ohe = pd.get_dummies(train_data_variant["Variation"], prefix="Var")
print(var_ohe.head())

   Var_1_2009trunc  Var_2010_2471trunc  Var_256_286trunc  Var_3' Deletion  \
0            False               False             False            False   
1            False               False             False            False   
2            False               False             False            False   
3            False               False             False            False   
4            False               False             False            False   

   Var_385_418del  Var_422_605trunc  Var_533_534del  Var_534_536del  \
0           False             False           False           False   
1           False             False           False           False   
2           False             False           False           False   
3           False             False           False           False   
4           False             False           False           False   

   Var_550_592del  Var_560_561insER  ...  Var_Y87N  Var_Y901C  Var_Y931C  \
0           False             Fals

#### Class

In [29]:
class_ohe = pd.get_dummies(train_data_variant["Class"], prefix="Class")
print(class_ohe.head())

   Class_1  Class_2  Class_3  Class_4  Class_5  Class_6  Class_7  Class_8  \
0     True    False    False    False    False    False    False    False   
1    False     True    False    False    False    False    False    False   
2    False     True    False    False    False    False    False    False   
3    False    False     True    False    False    False    False    False   
4    False    False    False     True    False    False    False    False   

   Class_9  
0    False  
1    False  
2    False  
3    False  
4    False  


### Target Encoding (Smoothed)

Safer version of response encoding

In [35]:
def target_encode(col, target, min_samples=100):
    mean = target.mean()
    agg = train_data_variant.groupby(col)[target.name].agg(["mean", "count"])
    smooth = (agg["count"] * agg["mean"] + min_samples * mean) / (agg["count"] + min_samples)
    return train_data_variant[col].map(smooth)

train_data_variant["Gene_target"] = target_encode("Gene", train_data_variant["Class"])
train_data_variant["Var_target"] = target_encode("Variation", train_data_variant["Class"])

print(train_data_variant[["Gene_target", "Var_target"]].head())

   Gene_target  Var_target
0     4.332528    2.816504
1     4.244683    4.342429
2     4.244683    4.342429
3     4.244683    4.352330
4     4.244683    4.362231


### Binary Encoding (compact alternative to OHE)

Using category_encoders:

In [36]:
!pip install category_encoders


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
  Using cached category_encoders-2.9.0-py3-none-any.whl.metadata (7.9 kB)
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
  Using cached statsmodels-0.14.6-cp314-cp314-win_amd64.whl.metadata (9.8 kB)
Using cached category_encoders-2.9.0-py3-none-any.whl (85 kB)
Using cached patsy-1.0.2-py2.py3-none-any.whl (233 kB)
Using cached statsmodels-0.14.6-cp314-cp314-win_amd64.whl (9.6 MB)

   ---------------------------------------- 0/3 [patsy]
   ---------------------------------------- 0/3 [patsy]
   ---------------------------------------- 0/3 [patsy]
   ---------------------------------------- 0/3 [patsy]
   ---------------------------------------- 0/3 [patsy]
   ---------------------------------------- 0/3 [patsy]
   ---------------------------------------- 0/3 [patsy]
   ---------------------------------------- 0/3 [patsy]
   ---------------------------------------- 0/3 [patsy]
   ------------

In [39]:
train_data_text = pd.read_csv(data_dir / 'training_text', sep=r'\|\|', engine='python', names=["ID", "TEXT"], skiprows=1)
train_data_variant = pd.read_csv(data_dir / 'training_variants.zip')
test_data_text = pd.read_csv(data_dir / 'test_text', sep=r'\|\|', engine='python', names=["ID", "TEXT"], skiprows=1)
test_data_variant = pd.read_csv(data_dir / 'test_variants.zip')

In [41]:
from category_encoders import BinaryEncoder

be = BinaryEncoder()
gene_binary_encoded = be.fit_transform(train_data_variant[["Gene"]])
variation_binary_encoded = be.fit_transform(train_data_variant[["Variation"]])
class_binary_encoded = be.fit_transform(train_data_variant[["Class"]])

print(gene_binary_encoded.head())
print(variation_binary_encoded.head())
print(class_binary_encoded.head())

   Gene_0  Gene_1  Gene_2  Gene_3  Gene_4  Gene_5  Gene_6  Gene_7  Gene_8
0       0       0       0       0       0       0       0       0       1
1       0       0       0       0       0       0       0       1       0
2       0       0       0       0       0       0       0       1       0
3       0       0       0       0       0       0       0       1       0
4       0       0       0       0       0       0       0       1       0
   Variation_0  Variation_1  Variation_2  Variation_3  Variation_4  \
0            0            0            0            0            0   
1            0            0            0            0            0   
2            0            0            0            0            0   
3            0            0            0            0            0   
4            0            0            0            0            0   

   Variation_5  Variation_6  Variation_7  Variation_8  Variation_9  \
0            0            0            0            0            

### Label Encoding (embeddings)

In [44]:
from sklearn.preprocessing import LabelEncoder

le_gene = LabelEncoder()
train_data_variant["Gene_idx"] = le_gene.fit_transform(train_data_variant["Gene"])

le_var = LabelEncoder()
train_data_variant["Var_idx"] = le_var.fit_transform(train_data_variant["Variation"])

le_class = LabelEncoder()
train_data_variant["Class_idx"] = le_class.fit_transform(train_data_variant["Class"])

print(train_data_variant[["Gene_idx", "Var_idx", "Class_idx"]].head())

   Gene_idx  Var_idx  Class_idx
0        85     2629          0
1        39     2856          1
2        39     1897          1
3        39     1667          2
4        39     1447          3


### TF-IDF for text

In [48]:
import re
def clean_text(text):
    """Basic text cleaning: lowercase, remove special chars, remove stopwords."""
    if isinstance(text, str):
        text = re.sub('[^a-zA-Z0-9\n]', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        text = text.lower()
        return ' '.join([w for w in text.split() if w not in STOP_WORDS])
    return ""


def apply_text_cleaning(df):
    """Apply text cleaning to all text columns."""
    df = df.copy()
    
    df['TEXT'] = df['TEXT'].fillna("")
    df['Gene'] = df['Gene'].fillna("Unknown")
    df['Variation'] = df['Variation'].fillna("Unknown")
    
    df['TEXT'] = df['TEXT'].apply(clean_text)
    
    return df

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

print(train_data_text["TEXT"].head())

train_data_text['TEXT'] = train_data_text['TEXT'].fillna("")

tfidf = TfidfVectorizer(max_features=5000)
text_tfidf = tfidf.fit_transform(train_data_text["TEXT"])

print(text_tfidf.shape)

0    Cyclin-dependent kinases (CDKs) regulate a var...
1     Abstract Background  Non-small cell lung canc...
2     Abstract Background  Non-small cell lung canc...
3    Recent evidence has demonstrated that acquired...
4    Oncogenic mutations in the monomeric Casitas B...
Name: TEXT, dtype: object
(3321, 5000)


In [52]:
print(text_tfidf)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4033623 stored elements and shape (3321, 5000)>
  Coords	Values
  (0, 1254)	0.493770181990411
  (0, 1351)	0.0118142286243278
  (0, 2579)	0.005346817406868727
  (0, 887)	0.046973715249330354
  (0, 3873)	0.006044752146458597
  (0, 4848)	0.008322457854230483
  (0, 3213)	0.33069991950777655
  (0, 1970)	0.009393712990778897
  (0, 892)	0.006536635047222687
  (0, 3622)	0.006216766969374531
  (0, 3268)	0.012221806610042626
  (0, 558)	0.05971890435143049
  (0, 3233)	0.009708749341664838
  (0, 4583)	0.3980937419667073
  (0, 2626)	0.006463714552015465
  (0, 1927)	0.03296283788503981
  (0, 4920)	0.0364077434256911
  (0, 3133)	0.0179555507378269
  (0, 342)	0.0046270503674904016
  (0, 2138)	0.014453890728652094
  (0, 680)	0.015910675024234572
  (0, 2287)	0.01340160457983204
  (0, 487)	0.3354018140979345
  (0, 2578)	0.03953973279230434
  (0, 348)	0.018438330272471378
  :	:
  (3320, 2110)	0.001781047180288865
  (3320, 4765)	0.00197254597479